In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical events Dx (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

UNION

-- Pharmacy events Dx (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}';


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (Tx universe for visit counting)
--   • Captures Elaprase-coded treatment from medical NDC, medical procedures, and paid pharmacy NDC
--   • Window: 2020-08-01 → ${end_date}
--   • Includes CODE field to retain NDC/procedure provenance
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical events Tx via NDC (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

UNION

-- Medical events Tx via procedures (NPI = rendering_npi only)
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

UNION

-- Pharmacy events Tx via NDC (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}';


-- =============================================================================
-- STEP 3: Tx CLAIMS IN ELIGIBILITY WINDOW (2y/refresh)
--   • Subset of all_tx_claims restricted to 2023-08-01 → ${end_date}
--   • Used ONLY to determine cohort eligibility (not for visit counting tiers)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND '${end_date}';


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
--   Builds eligible_patients using Dx evidence (≥2 dates) + Tx evidence in 2y window.
-- =============================================================================

-- 4A) Specified Dx requirement: ≥2 distinct E761 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e761_patients_1dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 1;

-- 4B) Specified cohort: specified Dx + any Tx in 2y/refresh window
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_1dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

-- 4C) Incremental Dx requirement: ≥2 distinct E763 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e763_patients_1dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 1;

-- 4D) Elaprase-coded Tx requirement for incremental eligibility (2y/refresh window)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');

-- 4E) Incremental cohort: incremental Dx + Elaprase-coded tx in 2y window + exclude specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_1dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- 4F) Final eligible cohort
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- Provider inclusion list (INDIVIDUAL NPIs only)
--   • Used to restrict HCPs considered for primary assignment
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
SELECT DISTINCT npi
FROM com_raw.kom_providers
WHERE provider_type = 'INDIVIDUAL'
  AND (
    PRIMARY_SPECIALTY NOT IN (
      'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
      'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
      'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
      'Radiology','Urology'
    )
    OR SECONDARY_SPECIALTY IN (
      'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
      'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
      'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
      'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
      'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
      'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
      'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
      'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
      'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
      'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
    )
  );


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
--   • Combines Dx + Tx events (visit dates) for eligible patients only
--   • Filters to included INDIVIDUAL NPIs (cohort_3_learnings)
--   • NOTE: As written, this excludes NULL NPI rows (because of the IN filter)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
SELECT *
FROM (
  -- -- Dx claims
  -- SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  -- FROM all_dx_claims
  -- WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  -- UNION

  -- Tx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)
-- WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


-- =============================================================================
-- Provider inclusion list (INDIVIDUAL NPIs only)
--   • Used to restrict HCPs considered for primary assignment
-- =============================================================================


In [0]:
select count(distinct patient_id), count(distinct npi) from all_patient_claims

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims_hco_min AS
SELECT DISTINCT
    apc.PATIENT_ID,
    apc.NPI     AS HCP_NPI,
    ref.HCO_NPI AS HCO_NPI,
    ref.HCO_NAME AS HCO_NAME
FROM all_patient_claims apc
LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 ref
  ON apc.NPI = ref.HCP_NPI;


In [0]:
select count(distinct patient_id), count(distinct hcp_npi), count(distinct hco_npi)
from all_patient_claims_hco_min

In [0]:
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty bucket (reporting label)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Tier 1: specialty priority (lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: visit counts (Dx + Tx combined)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference-only breakdowns (not used in ranking)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: most recent visit date
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;

In [0]:
CREATE OR REPLACE TEMP VIEW primary_hcp_with_hco AS
SELECT
    ph.*,
    ref.HCO_NPI,
    ref.HCO_NAME
FROM primary_hcp ph
LEFT JOIN (
    SELECT DISTINCT
      HCP_NPI,
      HCO_NPI,
      HCO_NAME
    FROM cmpa_insights_internal_schema.reference_file_0109
) ref
  ON ph.PRIMARY_HCP_NPI = ref.HCP_NPI;


In [0]:
select count(distinct patient_id), count(distinct primary_hcp_npi), count(distinct hco_npi)
from primary_hcp_with_hco